<a href="https://colab.research.google.com/github/ChaikaOlga/ChaikaOlga/blob/main/test_task_chaika.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Задача: создать прогностическую модель рисков беременных**

In [3]:
!pip install phik -q
!pip install catboost -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.7/679.7 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 11.3 MB/s eta 0:00:00


In [4]:
# необходимые импорты
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import phik

from sklearn.model_selection import (train_test_split,
                                     GridSearchCV)

from sklearn.preprocessing import (OneHotEncoder,
                                   StandardScaler)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from sklearn.tree import DecisionTreeClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (confusion_matrix,
                             make_scorer,
                             fbeta_score)


from sklearn.datasets import make_classification

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [6]:
# константы
RANDOM_STATE = 42
TEST_SIZE = 0.25

# Загрузка данных

In [7]:
try:
  data = pd.read_csv('/content/drive/MyDrive/data.csv')
except:
  data = pd.read_csv('https://drive.google.com/file/d/1--FeiK0Yo4GARky8s3OjujnXgOeicO1V/view?usp=sharing')
data.head(10)

,Age,SystolicBP,DiastolicBP,BS,BodyTemp,HeartRate,RiskLevel
0,25,130,80,15.00,98.0,86,high risk
1,35,140,90,13.00,98.0,70,high risk
2,29,90,70,8.00,100.0,80,high risk
3,30,140,85,7.00,98.0,70,high risk
4,35,120,60,6.10,98.0,76,low risk
5,23,140,80,7.01,98.0,70,high risk
6,23,130,70,7.01,98.0,78,mid risk
7,35,85,60,11.00,102.0,86,high risk
8,32,120,90,6.90,98.0,70,mid risk
9,42,130,80,18.00,98.0,70,high risk


In [ ]:
# общая информация
data.info()

Данные в файле соответствуют описанию. Пропуски отсутствуют. Тип данных оответсвует содержанию столбцов.

# Предобработка данных

Переименуем названия столбцов, проверим датасет на пропуски, на явные и неявные дубликаты.

In [ ]:
data.columns = ['age', 'systolic_bp', 'diastolic_bp', 'bs',
                'body_temp', 'heart_rate', 'risk_level']

data.sample(5)

In [ ]:
# пропуски
data.isna().sum()

In [ ]:
# уникальные значения
for i in data.columns:
  print('\033[1m' + i + '\033[0m', data[i].sort_values().unique())

Никаких опечаток нет.
Для удобства переведём температуру тела из Фаренгейта в градусы Цельсии

In [ ]:
data['body_temp'] = round((data['body_temp'] - 32)*5/9, 1)
data['body_temp'].sort_values().unique()

In [ ]:
# дубликаты в строках
data.duplicated().sum()

Обнаружены 562 строки, дублирующие данные. Удалить эти данные не можем, тк они составляют почти половину датасета. Есть предположение, что это одни и те же пациентки, отметившиеся в разное время. Пока оставим их.

In [ ]:
data[data.duplicated
 (keep=False)].sort_values(
     ['age', 'systolic_bp', 'diastolic_bp', 'bs']
     ).head(10)


**Вывод:** исправили названия столбцов в датасете;

проверили данные на пропуски;

проверили на явные и неявные дубликаты.

# Исследовательский анализ данных

Построим графики для количественных признаков.

In [ ]:
num_cols = data.select_dtypes(include=['int64', 'float64']).columns.tolist()

n = len(num_cols)
fig, axes = plt.subplots(n, 2, figsize=(10, 3*n), squeeze=False)

for i, col in enumerate(num_cols):
  # гистограмма
  axes[i, 0].hist(data[col], bins=20, color='skyblue', edgecolor='black')
  axes[i, 0].set_title(f'Гистограмма: {col}', pad=20)
  axes[i, 0].set_xlabel('Значения')
  axes[i, 0].set_ylabel('Частота')
  axes[i, 0].grid(alpha=0.4)

  # диаграмма размаха
  axes[i, 1].boxplot(data[col].dropna(),
                     vert=False,
                     patch_artist=True,
                     boxprops=dict(facecolor='lightgreen'))
  axes[i, 1].set_title(f'Диаграмма размаха: {col}', pad=20)
  axes[i, 1].set_xlabel('Значения')
  axes[i, 1].grid(alpha=0.4)

plt.tight_layout()
plt.show()

# описательная статистика
display(data[num_cols].describe())

Возраст распределен нормально со смещением влево. 70 лет выбивается из общей картины, но с ЭКО беременность вполне возможна.

3 четверти женщин имеют систолическое давление до 120 мм рт. ст. Имеется выброс в 160 мм рт. ст.

Половина женщин имеет диастолическое давление до 80 мм рт. ст. Выбросов в измерениях нет.

Содержание глюкозы в крови имеет много выбросов, но аномальными они не считаются. У беременных иногда наблюдается гестационный сахарный диабет, отсюда эти значения. Половина женщин имеет показание до 7,5 мМоль/л.

Температура тела в основном до 36.7, но есть и заболевшие женщины с температурой выше. Значения не аномальны.

Частота сердцебиения имеет одно аномальное значение - 7 ударов в мин. Надо будет посмотреть поближе.

In [ ]:
# срез женщин старше 65 лет
data.query('age>65')


70 летняя женщина такая одна. Удалим её.

In [ ]:
data = data.loc[data['age']<70]

In [ ]:
# срез систолического давления более 140 мм. рт. ст.
data.query('systolic_bp>140')

Видим, что это дублированные строки, составляют 0,01 от всех данных. Удалим их

In [ ]:
data = data.query('systolic_bp<160')
data.head(5)

In [ ]:
# срез пациенток с пульсом 7 уд/мин
data.query('heart_rate==7')

Наблюдаем 1 дубль одной пациентки с пульсом 7 ударов в минуту. Удалим их.

In [ ]:
data = data.query('heart_rate>7')
data.sample(5)

Посмотрим, как влияет уровень сахара на все признаки с учетом рисков. Построим диаграмму рассеяния, цвет точек отображают риски.

In [ ]:
for category in num_cols:
    g = sns.PairGrid(data,
                     hue = 'risk_level',
                     x_vars=[category],
                     y_vars=['bs'],
                     height=5,
                     aspect=1.5,
                     palette='Set2'
                    )
    g.map(sns.scatterplot)
    g.add_legend()
    g.fig.suptitle(f'Risk vs {category}', y=1.02)

    plt.show()

Наблюдаем, как уровень глюкозы более 8 мМоль/л разделяет данные. У многих женщин наблюдается высокий риск.

Добавим бинарный флаг `bs_high` (глюкоза > 8), но исходное значение `bs` оставим непрерывным, чтобы не потерять информацию.

In [ ]:
# добавляем бинарный признак

data['bs_high'] = (data['bs'] > 8).astype(int)

data.sample(5)

Добавиим еще один признак - разницу между систолическим и диастолическим давлением

In [ ]:
data['pulse_pressure'] = data['systolic_bp'] - data['diastolic_bp']
data.head()

Посмотрим на графике распределение целевого признака.

In [ ]:
# график с уровнем риска
plt.figure(figsize=(10, 5))
sns.countplot(x='risk_level', data=data, hue='risk_level', palette='viridis', legend=False)
plt.title('Распределение уровня риска')
plt.xlabel('Уровень риска')
plt.ylabel('Количество наблюдений')

# добавим значения на столбцы
for i, count in enumerate(data['risk_level'].value_counts().sort_index()):
    plt.text(i, count + 5, str(count), ha='center', fontsize=12)

plt.grid(True, alpha=0.3)
plt.show()

# статистика по уровням
class_distribution = data['risk_level'].value_counts().sort_index()
print("\033[1m Уровень риска, количество наблюдений:\033[0m")
print(class_distribution)
print(f"\033[1m Уровень риска в долях:\033[0m")
print((class_distribution / len(data)).round(3))

26% пациенток с высоким уровнем риска, 40% с низким, 34% со средним уровнем.

**Вывод:** удалили выбросы из данных в признаках: возраст, систолическое давление, и аномальное значение в признаке пульс.

Выделили уровень глюкозы в категориальный признак.

# Корреляционный анализ

Проведём корреляционный анализ признаков в количественной шкале.

In [ ]:
# карта корреляции heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(data.phik_matrix(interval_cols = [
    'age', 'systolic_bp', 'diastolic_bp', 'bs', 'body_temp', 'heart_rate']),
            annot=True,
            fmt='.2f',
            cmap='coolwarm')

plt.title('Тепловая карта data',
          fontsize=16)
plt.xlabel('Столбцы датасета', fontsize=14)
plt.ylabel('Столбцы датасета', fontsize=14)
plt.show()

Видим высокую корреляцию между диастолическим давлением и систолическим, диастолическим и пульсом (corr=0.78, 0.75), содержание глюкозы в крови и возрастом пациентки (corr=0.73).


Построим карту корреляции для количественных признаков. Будем использовать метод Спирмена, тк он подходит для выборок более 30 наблюдений, более чувствителен к сильным корреляциям - лучше улавливает монотонные связи.

In [ ]:
num_cols = ['age', 'systolic_bp', 'diastolic_bp', 'bs', 'body_temp',
            'heart_rate', 'bs_high', 'pulse_pressure']

# карта корреляции для колич признаков
plt.figure(figsize=(10, 8))
sns.heatmap(data[num_cols].corr(method='spearman'),
                annot=True, fmt='.2f',cmap='coolwarm')

plt.title('Тепловая карта: корреляция количественных столбцов data',
          fontsize=16)
plt.xlabel('Столбцы датасета', fontsize=14)
plt.ylabel('Столбцы датасета', fontsize=14)
plt.show()

Мультиколлинеарность не наблюдается.

**Вывод:** Провели корреляционный анализ данных. Использовали методы Фи для категориальных данных и метод Спирмена для количественных. Мультиколлинеарность в данных отсутствует.

# Подготовка к обучению

Закодируем целевой признак:
- низкий уровень = 0
- средний уровень = 1
- высокий уровень = 2

In [ ]:
# кодирование целевого признака
data.loc[data['risk_level']=='low risk', 'risk_level'] = 0
data.loc[data['risk_level']=='mid risk', 'risk_level'] = 1
data.loc[data['risk_level']=='high risk', 'risk_level'] = 2

data.sample(5)


In [ ]:
# смена типа данных в таргете
data['risk_level'] = data['risk_level'].astype('int')

data.info()

Разделим выборку на тренировочную и тестовую в соотношении 3:1

In [ ]:
# входные признаки
X = data.drop('risk_level', axis=1)
# целевой признак
y = data['risk_level']

# разбиение данных
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = TEST_SIZE,
    random_state = RANDOM_STATE,
    stratify = y
)

display(X_train.head())

Используем StandardScaler внутри ML-пайплайнов, чтобы исключить утечку данных при кросс-валидации.

In [ ]:
# предобработка
preprocess_numeric = ColumnTransformer(
    transformers=[('scaler', StandardScaler(), num_cols)],
    remainder='drop'
)


# Обучение моделей

Обучим 5 моделей в одном sklearn `Pipeline` с общей предобработкой и единым `GridSearchCV`, чтобы не дублировать код и исключить утечки данных:
- Logistic Regression
- Decision Tree
- RandomForest
- LGBM
- CatBoost

Метрика — F2 (β=2), так как нам важны и Precision, и Recall; приоритет — не пропустить высокий риск.

In [ ]:
# Единый Pipeline + GridSearchCV для всех моделей
model_pipeline = Pipeline([
    ('preprocess', preprocess_numeric),
    ('model', LogisticRegression(random_state=RANDOM_STATE, max_iter=1000))
])

# метрика F2
f2_scorer = make_scorer(fbeta_score, beta=2, average='weighted')

param_grid = [
    {
        'model': [LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)],
        'model__C': [0.001, 0.01, 0.1],
        'model__penalty': ['l1'],
        'model__solver': ['saga'],
        'model__class_weight': [None, 'balanced', {0: 1, 1: 2, 2: 5}]
    },
    {
        'model': [DecisionTreeClassifier(random_state=RANDOM_STATE)],
        'model__max_depth': [3, 15, 21, 26],
        'model__min_samples_split': [2, 5, 7],
        'model__min_samples_leaf': [1, 3, 5],
        'model__max_features': ['sqrt', 'log2', 0.6, 0.8],
        'model__criterion': ['gini', 'entropy']
    },
    {
        'model': [RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)],
        'model__criterion': ['gini'],
        'model__n_estimators': [135, 140, 200],
        'model__max_depth': [15, 17, 20],
        'model__min_samples_split': [2, 3, 5],
        'model__min_samples_leaf': [1, 2, 3],
        'model__bootstrap': [True]
    },
    {
        'model': [LGBMClassifier(random_state=RANDOM_STATE)],
        'model__n_estimators': [120, 135, 150],
        'model__max_depth': [13, 15, 17],
        'model__num_leaves': [20, 31, 50],
        'model__learning_rate': [0.01, 0.05, 0.1]
    },
    {
        'model': [CatBoostClassifier(random_state=RANDOM_STATE, verbose=0)],
        'model__depth': [4, 6, 8],
        'model__learning_rate': [0.03, 0.1, 0.3],
        'model__n_estimators': [100, 150]
    }
]

model_search = GridSearchCV(
    estimator=model_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring=f2_scorer,
    n_jobs=-1,
    verbose=1
)

model_search.fit(X_train, y_train)

# агрегированные результаты по моделям
cv_results = pd.DataFrame(model_search.cv_results_)
model_scores = (
    cv_results
    .assign(model_name=lambda df: df['param_model'].apply(lambda m: type(m).__name__),
            f2_score=lambda df: df['mean_test_score'].round(4))
    .groupby('model_name')['f2_score']
    .max()
    .reset_index()
    .sort_values('f2_score', ascending=False)
)

display(model_scores)

print("=== Лучшая модель ===")
print(type(model_search.best_estimator_.named_steps['model']).__name__)
print(model_search.best_params_)
print(f"Лучшая f2 мера: {model_search.best_score_:.4f}")

**Вывод:** Все 5 моделей обучаются в одном пайплайне с общей предобработкой; GridSearchCV подбирает гиперпараметры и выбирает лучшую по F2. Таблица выше `model_scores` показывает максимальную F2 для каждой модели.

Лучшая модель **LGBMClassifier** с гиперпараметрами  'model__learning_rate': 0.1, 'model__max_depth': 13, 'model__n_estimators': 135, 'model__num_leaves': 31}, с метрикой = 0.8065


# Тестирование модели

In [ ]:
# лучшая модель из общего GridSearchCV
best_model = model_search.best_estimator_

# предсказания на тестовых данных (pipeline сам масштабирует при необходимости)
y_pred_test = best_model.predict(X_test)

# метрики на тестовых данных
test_f2 = fbeta_score(y_test, y_pred_test, beta=2, average='weighted')
test_accuracy = accuracy_score(y_test, y_pred_test)

print("МЕТРИКИ НА ТЕСТОВЫХ ДАННЫХ:")
print(f"f2_score: {test_f2:.4f}")
print(f"Accuracy: {test_accuracy:.4f}")

**Метрика на тестовых данных = 0,84.**


Построим матрицы ошибок на тренировочной и тестовой выборках.

In [ ]:
y_pred_train = best_model.predict(X_train)
y_pred_test = best_model.predict(X_test)

cm_train = confusion_matrix(y_train, y_pred_train)
cm_test = confusion_matrix(y_test, y_pred_test)

# Матрицы ошибок на train и test
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# Train матрица
sns.heatmap(cm_train, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Матрица ошибок (Train)')
axes[0].set_ylabel('Истинный риск')
axes[0].set_xlabel('Предсказанный риск')

# Test матрица
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues', ax=axes[1])
axes[1].set_title('Матрица ошибок (Test)')
axes[1].set_ylabel('Истинный риск')
axes[1].set_xlabel('Предсказанный риск')

plt.tight_layout()
plt.show()


**На тестовых данных:**
Модель по 62 верно предсказала высокий уровень риска, 63 раза - средний и 85 раз - низкий уровень.

4 раза ошиблась - неверно дала предсказание на высокий риск. 4 пациентки оказались под угрозой.

14 раз модель ошиблась, недооценив средний уровень риска и 7 раз переоценила риск.

16 раз ошиблась, переоценив низкий уровень риска.

**На тренировочных данных:**
Модель ошибалась 5 раз, недооценив высокий риск, 20 раз переоценила низкие риски.
17 раз недооценила средний риск, 12 раз переоценила.

# Выводы



Перед нами стояла задача построить модель машинного обучения для прогноза рисков беременных.
Мы применили 5 моделей:
- Logistic Regression
- Decision Tree
- RandomForest
- LGBM
- CatBoost

Лучшей моделью оказалась **LGBMClassifier** с гиперпараметрами  {'model__learning_rate': 0.1, 'model__max_depth': 13, 'model__n_estimators': 135, 'model__num_leaves': 31}, с метрикой fbeta = 0.8065

На тестовых данных получили метрику = 0,84.

Эту модель можно использовать НЕ для диагноза, а для:
1. Сортировки очереди: "Кому проверить в первую очередь?"
2. Напоминание врачу: "Обратить внимание на этих пациенток"
3. Групповой скрининг: "В этой поликлинике выше % рисков"